# A Linear Autoencoder Finds PCA's Subspace, Not Its Axes

**Topic:** Autoencoders and PCA &nbsp;·&nbsp; **Difficulty:** Hard

A linear autoencoder trained with squared error recovers essentially the same
subspace as PCA. It does *not* recover the same basis: nothing in the
objective prefers orthogonal directions or orders them by variance, so the learned weights
look nothing like the principal components even when the reconstruction is
identical.

Compare them the right way: not weight-by-weight, but by the subspace they
project onto.

## The data (created in the next cell)

- `X` `(800, 20)`, centred and divided by its global standard deviation,
  with 4 dominant directions.
- Bottleneck size `k = 4`.

## Your task

With **NumPy only**:

1. Fit PCA by `eigh` on the covariance, and take the top-4 eigenvectors as
   `Vp` `(20, 4)`.
2. Train a linear autoencoder `X -> X @ W_enc @ W_dec` by full-batch gradient
   descent on the mean over the 800 rows of the squared residual norm,
   `(1/n) * sum_i ||x_i @ W_enc @ W_dec - x_i||^2`: **60,000 steps**, learning
   rate 0.02, weights drawn from `np.random.default_rng(33).normal(0, 0.1, ...)`,
   shapes `(20, 4)` and `(4, 20)`. It needs that many steps: the reconstruction
   error converges long before the subspace does.
3. Compare:
   - `mse_pca`, `mse_ae`: reconstruction errors,
   - `weight_diff`: the Frobenius distance `||Wn - Vp||_F` between `W_enc`'s
     unit-norm columns and `Vp`, taken in column order (this will be large),
   - `subspace_diff`: `||P_ae - P_pca||_F` where each `P` is the orthogonal
     projector onto the respective column space (this will be small).
4. Set `answers` to a dict with those four values plus
   `"principal_angles_max"`, the largest principal angle in radians between the
   two subspaces.

## Evaluation

Compared to the reference within 1e-4 times (1 plus the reference's magnitude). The evaluation cell
checks the reconstruction errors agree while the raw weights do not, and that the
subspaces coincide.

In [ ]:
import numpy as np

rng = np.random.default_rng(3102)
n, d, k = 800, 20, 4
B = rng.normal(0, 1, (k, d))
Z = rng.normal(0, 1, (n, k)) * np.array([4.0, 3.0, 2.0, 1.5])
X = Z @ B + rng.normal(0, 0.25, (n, d))
X = X - X.mean(axis=0)
X = X / X.std()                       # keep the plain-GD step size stable
print(f"X {X.shape}, {k} dominant directions, bottleneck k = {k}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("mse_pca", [], 0.001443220057661617, -0.00024164990139532817, 0.0001),
    ("mse_ae", [], 0.001443232335238948, -0.00024165195712850317, 0.0001),
    ("weight_diff", [], 2.6755328044302944, -0.4479859013448781, 0.0001),
    ("subspace_diff", [], 0.017268446977100504, -0.0028913944807751474, 0.0001),
    ("principal_angles_max", [], 0.01024892932458207, -0.0017160603800821274, 0.0001),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "subspace_diff" in answers:
    _same_err = abs(answers["mse_ae"] - answers["mse_pca"]) < 0.02 * answers["mse_pca"]
    _diff_w = answers["weight_diff"] > 0.5
    _same_sub = answers["subspace_diff"] < 0.05
    print(f"  {'same reconstruction':<24} {'ok' if _same_err else 'wrong'}  "
          f"({answers['mse_ae']:.5f} vs {answers['mse_pca']:.5f})")
    print(f"  {'different weights':<24} {'ok' if _diff_w else 'wrong'}  "
          f"(||W - V|| = {answers['weight_diff']:.3f})")
    print(f"  {'same subspace':<24} {'ok' if _same_sub else 'wrong'}  "
          f"(||P - P|| = {answers['subspace_diff']:.4f})")
    _ok = _ok and _same_err and _diff_w and _same_sub

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

The autoencoder's loss depends on `W_enc @ W_dec` only through the product, and
any invertible `4x4` matrix `M` gives the same reconstruction via
`(W_enc M)(M^-1 W_dec)`. Watch the two convergence rates: the reconstruction
error matches PCA within a few thousand steps, but the subspace takes an order of
magnitude longer, because the loss is nearly flat along the directions that still
need to rotate. So the solution is a whole equivalence class of bases
for one subspace, and gradient descent lands on an arbitrary member. PCA picks
the orthonormal, variance-ordered member. Comparing projectors, or equivalently
principal angles, is the only meaningful way to check two subspaces agree, and
it is the same check you need whenever comparing learned representations.

```python
import numpy as np

C = (X.T @ X) / (len(X) - 1)
w, V = np.linalg.eigh(C)
Vp = V[:, np.argsort(w)[::-1][:k]]

rng2 = np.random.default_rng(33)
We = rng2.normal(0, 0.1, (d, k))
Wd = rng2.normal(0, 0.1, (k, d))

n = len(X)
for _ in range(60000):
    R = X @ We @ Wd - X
    gWe = 2.0 * (X.T @ (R @ Wd.T)) / n
    gWd = 2.0 * ((X @ We).T @ R) / n
    We -= 0.02 * gWe
    Wd -= 0.02 * gWd

def mse(P):
    return float(np.mean((X @ P - X) ** 2))

mse_ae = mse(We @ Wd)
mse_pca = mse(Vp @ Vp.T)

Wn = We / np.linalg.norm(We, axis=0, keepdims=True)
weight_diff = float(np.linalg.norm(Wn - Vp))

Qa, _ = np.linalg.qr(We)
Pa = Qa @ Qa.T
Pp = Vp @ Vp.T
subspace_diff = float(np.linalg.norm(Pa - Pp))

sv = np.linalg.svd(Qa.T @ Vp, compute_uv=False)
angles = np.arccos(np.clip(sv, -1.0, 1.0))

answers = {
    "mse_pca": mse_pca,
    "mse_ae": mse_ae,
    "weight_diff": weight_diff,
    "subspace_diff": subspace_diff,
    "principal_angles_max": float(angles.max()),
}
```